# Big Query to download Ethereum data
-  The following code has been taken from the kaggle notebook as reference.
- A service account needs to be created.
- bigquery and db-mbs library needs to be installed.

In [31]:
%pip install db-dtypes


[notice] A new release of pip is available: 23.0 -> 23.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
from google.cloud import bigquery
import pandas as pd
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/dr.rubaiyatislam/Documents/big_query/keyfile.json"
client = bigquery.Client()

# Query by Allen Day, GooglCloud Developer Advocate (https://medium.com/@allenday)
query = """
SELECT 
  SUM(value/POWER(10,18)) AS sum_tx_ether,
  AVG(gas_price*(receipt_gas_used/POWER(10,18))) AS avg_tx_gas_cost,
  DATE(timestamp) AS tx_date
FROM
  `bigquery-public-data.crypto_ethereum.transactions` AS transactions,
  `bigquery-public-data.crypto_ethereum.blocks` AS blocks
WHERE TRUE
  AND transactions.block_number = blocks.number
  AND receipt_status = 1
  AND value > 0
GROUP BY tx_date
HAVING tx_date >= '2015-01-01' AND tx_date <= '2023-12-31'
ORDER BY tx_date
"""
query_job = client.query(query)

iterator = query_job.result(timeout=90)
rows = list(query_job)

# Transform the rows into a nice pandas dataframe
df = pd.DataFrame(data=[list(x.values()) for x in rows], columns=list(rows[0].keys()))

# Look at the first 10
df.tail(3)

,sum_tx_ether,avg_tx_gas_cost,tx_date
1952,7.666137e+05,0.002499,2023-02-19
1953,1.130323e+06,0.003400,2023-02-20
1954,8.396978e+05,0.002643,2023-02-21


In [28]:
df.head(3)

,sum_tx_ether,avg_tx_gas_cost,tx_date
0,6.176871e+06,0.000592,2017-10-16
1,1.109204e+07,0.000464,2017-10-17
2,1.459824e+07,0.000494,2017-10-18


# Block table and the attributes from 2015 to 2018
- The following query explores all the attributes of the block

In [ ]:
query1 = """ 
SELECT 
  timestamp,
  number AS block_number,
  hash 
  parent_hash
  nonce,
  miner,
  difficulty,
  size,
  gas_limit,
  gas_used,
  transaction_count
  base_fee_per_gas
  
FROM
  `bigquery-public-data.crypto_ethereum.blocks`
WHERE
  EXTRACT(YEAR FROM timestamp) >= 2015
  AND EXTRACT(YEAR FROM timestamp) <= 2018
ORDER BY timestamp
"""

In [24]:
# Execute the query and convert the results to a Pandas DataFrame
# There is an issue with the 'hash' attribute in the above query, it produces a BadRequest error
df2 = client.query(query1).to_dataframe()
df2.head(3)

BadRequest: 400 Syntax error: Expected end of input but got keyword HASH at [4:3]

Location: US
Job ID: 82776070-837b-450f-9909-40b4ec400075


- The following query works if we omit 'hash' attribute. 
- We need to fix the hash attribute error issue. 

In [20]:
# The following query does not have the 'hash' attributes to be queried. It Works!!!!
query2 = """ 
SELECT 
  timestamp,
  number AS block_number, 
  parent_hash
  nonce,
  miner,
  difficulty,
  size,
  gas_limit,
  gas_used,
  transaction_count
  base_fee_per_gas
  
FROM
  `bigquery-public-data.crypto_ethereum.blocks`
WHERE
  EXTRACT(YEAR FROM timestamp) >= 2015
  AND EXTRACT(YEAR FROM timestamp) <= 2018
ORDER BY timestamp
"""


In [29]:

# Execute the query and convert the results to a Pandas DataFrame
df2 = client.query(query2).to_dataframe()
df2.head()

In [30]:
# Exporting the dataframe to a text file:
df2.to_csv("/Users/dr.rubaiyatislam/Documents/big_query/block.txt",sep='\t')